# 01 — Eksplorasi Dataset (5 kelas)

Statistik foto multi-bean hasil anotasi Roboflow dan contoh citranya.

Pipeline: **anotasi Roboflow → crop (`crops_raw`) → augmentasi offline (`crops`) → 3 CNN → analisis**.

Dataset memakai 5 kelas granular: `good`, `black`, `chipped`, `dry_pods`, `insect_damage`.

**Strategi anotasi asimetris.** Sebagian foto dianotasi penuh (semua biji berlabel, termasuk `good`), sebagian hanya bijinya yang cacat. Notebook ini memisahkan keduanya, karena foto *partial* tidak menyumbang sampel `good` sama sekali.

## Setup

In [ ]:
# Bootstrap: tambahkan root proyek ke sys.path agar modul src/ & configs/ dapat diimpor.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from configs.base_config import (
    ANNOTATED_DATA_DIR,
    CLASS_COLORS,
    CLASS_DESCRIPTIONS,
    CLASS_NAMES,
    CLASS_NAMES_ID,
    CROPS_DIR,
    CROPS_RAW_DIR,
    GOOD_CLASS,
    SPLITS,
)
from src.analysis.style import apply_plot_style, class_palette
from src.training.imbalance import describe_imbalance

apply_plot_style()

IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')


def list_images(directory):
    """Daftar citra dalam folder (mengabaikan .gitkeep dan file lain)."""
    if not directory.is_dir():
        return []
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in IMAGE_EXTS)


pd.DataFrame(
    [
        {'kelas': n, 'indonesia': CLASS_NAMES_ID[n], 'definisi': CLASS_DESCRIPTIONS[n]}
        for n in CLASS_NAMES
    ]
).set_index('kelas')

## 1. Jumlah foto & label per split

In [ ]:
rows = []
for split in SPLITS:
    images = list_images(ANNOTATED_DATA_DIR / split / 'images')
    labels = sorted((ANNOTATED_DATA_DIR / split / 'labels').glob('*.txt'))
    rows.append({'split': split, 'foto': len(images), 'label': len(labels)})

df_splits = pd.DataFrame(rows).set_index('split')
print(df_splits.to_string())
print()
if df_splits['foto'].sum() == 0:
    print('data/annotated/ masih kosong — jalankan skill /import-dataset dulu.')
elif not (df_splits['foto'] == df_splits['label']).all():
    print('PERINGATAN: jumlah foto dan label tidak cocok di sebagian split.')

## 2. Distribusi 5 kelas pada anotasi (sebelum crop)

Dihitung langsung dari baris label YOLO, jadi angka ini adalah jumlah bounding box — target jumlah crop yang seharusnya dihasilkan notebook 02.

In [ ]:
per_split = {}
for split in SPLITS:
    counter = Counter()
    for label_path in sorted((ANNOTATED_DATA_DIR / split / 'labels').glob('*.txt')):
        for line in label_path.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            class_id = int(float(line.split()[0]))
            if 0 <= class_id < len(CLASS_NAMES):
                counter[CLASS_NAMES[class_id]] += 1
    per_split[split] = {name: counter.get(name, 0) for name in CLASS_NAMES}

df_anno = pd.DataFrame(per_split).T[CLASS_NAMES]
df_anno['total'] = df_anno.sum(axis=1)
df_anno.loc['TOTAL'] = df_anno.sum()
df_anno

In [ ]:
total = df_anno.loc['TOTAL', CLASS_NAMES]
if total.sum():
    share = (total / total.sum() * 100).round(1)
    print('Proporsi kelas (%):')
    print(share.to_string())
    print()
    print('Ketimpangan:', describe_imbalance(total.to_dict()))
    print()
    defect_total = total.drop(GOOD_CLASS).sum()
    print(f'good = {total[GOOD_CLASS]:,} biji | cacat (4 kelas) = {defect_total:,} biji')

## 3. Visualisasi distribusi kelas

In [ ]:
if df_anno.loc['TOTAL', CLASS_NAMES].sum():
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

    totals = df_anno.loc['TOTAL', CLASS_NAMES]
    axes[0].bar(CLASS_NAMES, totals.values, color=class_palette(CLASS_NAMES))
    axes[0].set_title('Jumlah biji per kelas (semua split)')
    axes[0].set_ylabel('jumlah bounding box')
    axes[0].tick_params(axis='x', rotation=30)
    for i, v in enumerate(totals.values):
        axes[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)

    split_rows = df_anno.drop(index='TOTAL')[CLASS_NAMES]
    bottom = None
    for name in CLASS_NAMES:
        axes[1].bar(
            split_rows.index, split_rows[name], bottom=bottom,
            label=name, color=CLASS_COLORS[name],
        )
        bottom = split_rows[name] if bottom is None else bottom + split_rows[name]
    axes[1].set_title('Komposisi kelas per split')
    axes[1].set_ylabel('jumlah bounding box')
    axes[1].legend(fontsize=8)

    fig.tight_layout()
    plt.show()

## 4. Foto dianotasi penuh vs sebagian

Foto yang **tidak** memuat satu pun label `good` hampir pasti dianotasi sebagian (hanya cacatnya). Pembagian ini perlu proporsional antar-split: kalau semua foto *fully annotated* menumpuk di train, split test tidak akan punya sampel `good` yang memadai.

In [ ]:
rows = []
for split in SPLITS:
    for label_path in sorted((ANNOTATED_DATA_DIR / split / 'labels').glob('*.txt')):
        counter = Counter()
        for line in label_path.read_text(encoding='utf-8').splitlines():
            if line.strip():
                class_id = int(float(line.split()[0]))
                if 0 <= class_id < len(CLASS_NAMES):
                    counter[CLASS_NAMES[class_id]] += 1
        rows.append(
            {
                'split': split,
                'foto': label_path.stem,
                'n_box': sum(counter.values()),
                'n_good': counter.get(GOOD_CLASS, 0),
                'anotasi': 'full' if counter.get(GOOD_CLASS, 0) else 'partial',
            }
        )

df_photos = pd.DataFrame(rows)
if not df_photos.empty:
    print(df_photos.groupby(['split', 'anotasi']).agg(
        n_foto=('foto', 'count'), n_box=('n_box', 'sum')
    ).to_string())
    print()
    print('Total foto:', len(df_photos))
    print(df_photos['anotasi'].value_counts().to_string())

## 5. Contoh citra multi-bean

In [ ]:
sample_dir = ANNOTATED_DATA_DIR / 'train' / 'images'
samples = list_images(sample_dir)[:4]

if samples:
    from PIL import Image

    fig, axes = plt.subplots(1, len(samples), figsize=(4 * len(samples), 4))
    axes = [axes] if len(samples) == 1 else list(axes)
    for ax, path in zip(axes, samples):
        with Image.open(path) as im:
            ax.imshow(im)
            ax.set_title(f'{path.stem[:18]}...\n{im.size[0]}x{im.size[1]}', fontsize=8)
        ax.axis('off')
    fig.tight_layout()
    plt.show()
else:
    print('Belum ada citra di', sample_dir)

## 6. Status crop (bila notebook 02 sudah dijalankan)

`crops_raw` = hasil crop mentah, `crops` = dataset siap latih (train sudah diaugmentasi offline).

In [ ]:
for label, root in (('crops_raw', CROPS_RAW_DIR), ('crops', CROPS_DIR)):
    rows = []
    for split in SPLITS:
        counts = {
            name: len(list((root / split / name).glob('*.jpg'))) for name in CLASS_NAMES
        }
        counts['total'] = sum(counts.values())
        rows.append({'split': split, **counts})
    frame = pd.DataFrame(rows).set_index('split')
    print(f'=== {label} ===')
    if frame['total'].sum() == 0:
        print('  belum ada — jalankan notebook 02 atau scripts.crop_dataset\n')
    else:
        print(frame.to_string(), '\n')